In [1]:
import io
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astroquery.ipac.ned import Ned
from astroquery.sdss import SDSS
from astroquery.eso import Eso
from astroquery.vizier import Vizier

from astropy.coordinates import SkyCoord
import astropy.units as u
import warnings
warnings.filterwarnings('ignore')

In [2]:
with open("host_galaxy_ascii.txt", "r") as f:
    lines = f.readlines()

start_idx = 0
for i, line in enumerate(lines):
    if "2004ef" in line:
        start_idx = i
        break

df = pd.read_csv(io.StringIO("".join(lines[start_idx:])), 
                 sep=r'\s+', 
                 header=None, 
                 engine='python')


def reconstruct_galaxy(row):
    parts = []
    for item in row[1:]:
        if str(item).lower() in ['sa', 'sb', 's', 's0', 'e', 'cd', 'cdots']:
            break
        parts.append(str(item))
    return " ".join(parts)

df['Full_Galaxy_Name'] = df.apply(reconstruct_galaxy, axis=1)

def clean_mag(val):
    if val == 'cdots' or pd.isna(val):
        return np.nan
    return float(str(val).split('(')[0])

print("Successfully extracted Galaxy Names:")
print(df[['Full_Galaxy_Name']])

df[['Full_Galaxy_Name']].to_csv("data/extracted_galaxies.csv", index=False)

Successfully extracted Galaxy Names:
    Full_Galaxy_Name
0          UGC 12158
1           NGC 6928
2          UGC 11816
3     MCG +03-22-020
4           FGC 175A
..               ...
108         NGC 1725
109         NGC 1080
110        ESO 478-6
111        PGC 34730
112         NGC 5728

[113 rows x 1 columns]


In [3]:
import os
import time
import pandas as pd
import numpy as np

from astroquery.ipac.ned import Ned
from astroquery.sdss import SDSS
from astroquery.eso import Eso
from astroquery.vizier import Vizier

from astropy.coordinates import SkyCoord
import astropy.units as u

# =========================
# Setup
# =========================
folder_path = "data/spectra"
os.makedirs(folder_path, exist_ok=True)

eso = Eso()
Vizier.ROW_LIMIT = 10

# =========================
# Helper
# =========================
def clean_filename(name):
    return name.replace(" ", "_").replace("+", "plus").replace("(", "").replace(")", "")

# =========================
# NED
# =========================
def query_ned_all_spectra(name):
    try:
        spectra = Ned.get_spectra(name)

        if spectra:
            fname = f"{folder_path}/{clean_filename(name)}_NED.fits"
            spectra[0][0].writeto(fname, overwrite=True, output_verify='silentfix')
            print(f"[NED] Success: {name}")
            return True

    except Exception as e:
        print(f"[NED] Failed: {name} ({e})")

    return False

# =========================
# SDSS (North)
# =========================
def query_sdss(pos, name):
    try:
        spectra = SDSS.get_spectra(coordinates=pos, radius=5*u.arcsec)

        if spectra:
            fname = f"{folder_path}/{clean_filename(name)}_SDSS.fits"
            spectra[0][0].writeto(fname, overwrite=True)

            print(f"[SDSS] Success: {name}")
            return True

    except Exception as e:
        print(f"[SDSS] Failed: {name} ({e})")

    return False

# =========================
# ESO (Southern) — FIXED (TAP)
# =========================
def query_eso_spectrum(pos, name):
    try:
        eso.ROW_LIMIT = 5

        table = eso.query_main(
            column_filters={
                'ra': f"between {pos.ra.deg - 0.01} and {pos.ra.deg + 0.01}",
                'dec': f"between {pos.dec.deg - 0.01} and {pos.dec.deg + 0.01}",
                'dp_type': "like '%SPECTRUM%'"
            }
        )

        if table is not None and len(table) > 0:

            print(f"[ESO] Found {len(table)} entries for {name}")

            dp_id = table['dp_id'][0]
            eso.retrieve_data(dp_id, destination=folder_path)

            print(f"[ESO] Success: {name}")
            return True

    except Exception as e:
        print(f"[ESO] Failed: {name} ({e})")

    return False

# =========================
# 6dF / 2dF (Southern surveys)
# =========================
def query_6df_2df(pos, name):
    try:
        # 6dF Galaxy Survey catalog
        result = Vizier.query_region(
            pos,
            radius=10*u.arcsec,
            catalog=["VII/259"]   # 6dFGS
        )

        if result:
            print(f"[6dF] Found match for {name} (manual spectrum download needed)")
            return True

        # 2dF Galaxy Redshift Survey
        result = Vizier.query_region(
            pos,
            radius=10*u.arcsec,
            catalog=["VII/250"]   # 2dFGRS
        )

        if result:
            print(f"[2dF] Found match for {name} (manual spectrum download needed)")
            return True

    except Exception as e:
        print(f"[6dF/2dF] Failed: {name} ({e})")

    return False

# =========================
# MAIN PIPELINE
# =========================
def download_host_spectra(name):

    print(f"\n--- Searching for {name} ---")

    # --- 1. NED ---
    if query_ned_all_spectra(name):
        return True

    # --- 2. Coordinates ---
    try:
        obj = Ned.query_object(name)
        pos = SkyCoord(obj['RA'][0], obj['DEC'][0], unit=(u.deg, u.deg))
    except Exception as e:
        print(f"[COORD] Failed: {name} ({e})")
        return False

    # --- 3. SDSS ---
    if query_sdss(pos, name):
        return True

    # --- 4. ESO ---
    if query_eso_spectrum(pos, name):
        return True

    # --- 5. 6dF / 2dF ---
    if query_6df_2df(pos, name):
        return True

    print(f"[FAIL] No spectra found for {name}")
    return False

# =========================
# RUN
# =========================
df_names = pd.read_csv('data/extracted_galaxies.csv')

for galaxy in df_names['Full_Galaxy_Name'].unique():
    download_host_spectra(galaxy)
    time.sleep(1.5)


--- Searching for UGC 12158 ---
[NED] Success: UGC 12158

--- Searching for NGC 6928 ---
[FAIL] No spectra found for NGC 6928

--- Searching for UGC 11816 ---
[NED] Success: UGC 11816

--- Searching for MCG +03-22-020 ---
[NED] Success: MCG +03-22-020

--- Searching for FGC 175A ---
[NED] Success: FGC 175A

--- Searching for NGC 958 ---
[NED] Success: NGC 958

--- Searching for 2MASS J14564320+0919366 ---
[FAIL] No spectra found for 2MASS J14564320+0919366

--- Searching for NGC 5304 ---
[NED] Success: NGC 5304

--- Searching for NGC 2811 ---
[NED] Success: NGC 2811

--- Searching for 2MASS J14593308+1640068 ---
[NED] Failed: 2MASS J14593308+1640068 (HTTPSConnectionPool(host='ned.ipac.caltech.edu', port=443): Read timed out. (read timeout=60))
[COORD] Failed: 2MASS J14593308+1640068 (HTTPSConnectionPool(host='ned.ipac.caltech.edu', port=443): Read timed out. (read timeout=60))

--- Searching for MCG +03-31-93 ---
[NED] Success: MCG +03-31-93

--- Searching for NGC 4059 ---
[SDSS] Succ